
# Scaling Laws And Empirical Science

Scaling laws are empirical summaries of a measured regime, not guarantees about all future models. This notebook fits a small synthetic scaling curve, adds bootstrap uncertainty, and uses simple compute accounting to connect the Kaplan-style `loss vs scale` picture to the Hoffmann or Chinchilla-style `fixed compute` critique [@scaling_laws_for_neural_language_models_2020; @training_compute_optimal_large_language_models_2022].



## Motivation

The attractive part of scaling laws is that a handful of measured points can look smooth on log-log axes. The dangerous part is the same: smooth curves tempt us to treat an empirical fit as a law of nature. Kaplan et al. showed that language-model loss often follows a power-law trend over a measured range [@scaling_laws_for_neural_language_models_2020]. Hoffmann et al. then showed that the apparent frontier also depends on how compute is split between parameters and data [@training_compute_optimal_large_language_models_2022].

The practical lesson is narrow and operational: fit the curve, measure uncertainty, account for compute, and do not pretend that a small local trend automatically extrapolates to a new regime.



## Hypothesis

If a synthetic loss curve really follows `L(N) = L_inf + A N^{-alpha}`, then a small fitting utility should recover `alpha` and `L_inf` within tolerance and a bootstrap should produce a narrow interval around the recovered exponent. If we then fit only a small local window, or move to a slightly different regime, the same fitting procedure should become much less trustworthy for frontier extrapolation.



## Baseline

The baseline is the naive story often told in slides: fit a straight line to a few `log(loss)` versus `log(scale)` points and extend it far beyond the measured window. That baseline ignores three things this notebook keeps explicit:

1. an irreducible loss floor;
2. uncertainty in the fitted exponent;
3. compute allocation between parameters and tokens.



## Metric

We track four diagnostics:

- fitted exponent `alpha`;
- fitted irreducible loss `L_inf`;
- bootstrap confidence interval for `alpha`;
- compute-accounting consistency under `C ~= 6 N D`, where `N` is parameters and `D` is tokens.

For the extrapolation critique, we also compare predicted loss at a much larger frontier scale against the known synthetic truth. A fit that is numerically tidy inside the observed window can still be badly wrong outside it.



## Mathematical derivation

We model loss as

`L(N) = L_inf + A N^{-alpha}`

where `N` is model scale, `L_inf` is the irreducible floor, `A > 0` is a coefficient, and `alpha > 0` is the scaling exponent. If `L_inf` were known, then subtracting it and taking logs gives

`log(L(N) - L_inf) = log A - alpha log N`.

This is ordinary linear regression in `x = log N` and `y = log(L - L_inf)`, with slope `-alpha` and intercept `log A`. In practice `L_inf` is not known, so we search over a small grid of admissible floors and keep the fit with the smallest log-space residual.

The compute proxy used throughout the dense-transformer scaling literature is

`C ~= 6 N D`.

Holding `C` fixed forces a tradeoff: larger models must usually see fewer tokens unless total compute also rises. That is the core reason a pure `loss vs parameter count` story is incomplete.


In [ ]:

from pathlib import Path
import sys
import math

import torch
import torch.nn as nn

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from llm_from_scratch.research.scaling import (
    bootstrap_power_law_fit,
    chinchilla_optimal_tokens,
    fit_power_law,
    parameters_for_compute_budget,
    power_law_losses,
    tokens_for_compute_budget,
    training_compute_flops,
)

torch.manual_seed(0)
torch.set_printoptions(precision=6, sci_mode=False)



## PyTorch implementation

The implementation surface is intentionally small:

- `power_law_losses()` evaluates `L_inf + A N^{-alpha}`;
- `fit_power_law()` recovers `A`, `alpha`, and `L_inf` from a small sweep;
- `bootstrap_power_law_fit()` resamples the sweep and refits the floor each time to quantify joint parameter uncertainty;
- `training_compute_flops()` plus the inverse helpers keep the `C ~= 6 N D` accounting explicit.

We start with a deterministic synthetic sweep so the notebook stays fast and the numerical checks remain reproducible.


In [ ]:

scales = torch.tensor([1024.0, 2048.0, 4096.0, 8192.0, 16384.0, 32768.0, 65536.0], dtype=torch.float64)
true_curve = {
    'coefficient': 18.0,
    'exponent': 0.37,
    'irreducible_loss': 1.45,
}
multiplicative_noise = torch.tensor([1.00, 1.02, 0.99, 1.01, 0.985, 1.015, 0.995], dtype=torch.float64)

noiseless_losses = power_law_losses(scales, **true_curve)
observed_losses = true_curve['irreducible_loss'] + (noiseless_losses - true_curve['irreducible_loss']) * multiplicative_noise

full_fit = fit_power_law(scales, observed_losses)
bootstrap = bootstrap_power_law_fit(
    scales,
    observed_losses,
    num_resamples=200,
    seed=0,
)

print('true exponent:', true_curve['exponent'])
print('fit exponent :', round(full_fit.exponent, 6))
print('fit floor    :', round(full_fit.irreducible_loss, 6))
print('bootstrap exponent CI (unconditional):', tuple(round(x, 6) for x in (bootstrap.exponent.lower, bootstrap.exponent.median, bootstrap.exponent.upper)))
print('bootstrap floor CI (unconditional)   :', tuple(round(x, 6) for x in (bootstrap.irreducible_loss.lower, bootstrap.irreducible_loss.median, bootstrap.irreducible_loss.upper)))
print('observed losses:')
print(observed_losses)
print('predicted losses:')
print(full_fit.predicted_losses)


In [ ]:

base_parameters = 70_000_000
base_tokens = 1_400_000_000
base_compute = training_compute_flops(base_parameters, base_tokens)

larger_model_parameters = 280_000_000
same_budget_tokens = tokens_for_compute_budget(larger_model_parameters, base_compute)
chinchilla_tokens = chinchilla_optimal_tokens(larger_model_parameters)
reconstructed_parameters = parameters_for_compute_budget(base_tokens, base_compute)

print(f'base compute budget: {base_compute:.3e} FLOPs')
print(f'same compute, {larger_model_parameters:,} params -> {same_budget_tokens:,.0f} tokens')
print(f'20 tokens/parameter heuristic for {larger_model_parameters:,} params -> {chinchilla_tokens:,.0f} tokens')
print(f'parameter inversion check at the base budget -> {reconstructed_parameters:,.0f} params')



## Numerical checks

The core checks are deliberately modest. We are not proving a universal law; we are checking that the synthetic fit recovers the known parameters and that the compute helpers are internally consistent.


In [ ]:

assert abs(full_fit.exponent - true_curve['exponent']) < 0.03
assert abs(full_fit.irreducible_loss - true_curve['irreducible_loss']) < 0.05
assert bootstrap.exponent.lower <= true_curve['exponent'] <= bootstrap.exponent.upper
assert abs(tokens_for_compute_budget(base_parameters, base_compute) - float(base_tokens)) < 1e-6
assert abs(reconstructed_parameters - float(base_parameters)) < 1e-6

print('All numerical checks passed.')



## Ablations

Two ablations matter here.

First, fit only the smallest four scales from the otherwise well-behaved synthetic sweep. The local window still looks smooth, but it can miss the irreducible floor and become overconfident about how much more loss will fall.

Second, build a two-regime synthetic curve. Even tiny residuals inside the early window do not justify extrapolating that local slope to a frontier regime if the mechanism has changed.


In [ ]:

frontier_scale = torch.tensor([1_048_576.0], dtype=torch.float64)

small_window_fit = fit_power_law(scales[:4], observed_losses[:4])
true_frontier_loss = float(power_law_losses(frontier_scale, **true_curve).item())
full_window_frontier = float(power_law_losses(frontier_scale, coefficient=full_fit.coefficient, exponent=full_fit.exponent, irreducible_loss=full_fit.irreducible_loss).item())
small_window_frontier = float(power_law_losses(frontier_scale, coefficient=small_window_fit.coefficient, exponent=small_window_fit.exponent, irreducible_loss=small_window_fit.irreducible_loss).item())

regime_losses = 1.40 + 7.0 * torch.pow(scales, -0.22) + 45.0 * torch.pow(scales, -0.60)
local_regime_fit = fit_power_law(scales[:4], regime_losses[:4])
full_regime_fit = fit_power_law(scales, regime_losses)
true_regime_frontier = float((1.40 + 7.0 * torch.pow(frontier_scale, -0.22) + 45.0 * torch.pow(frontier_scale, -0.60)).item())
local_regime_frontier = float(power_law_losses(frontier_scale, coefficient=local_regime_fit.coefficient, exponent=local_regime_fit.exponent, irreducible_loss=local_regime_fit.irreducible_loss).item())
full_regime_frontier = float(power_law_losses(frontier_scale, coefficient=full_regime_fit.coefficient, exponent=full_regime_fit.exponent, irreducible_loss=full_regime_fit.irreducible_loss).item())

print('single-law sweep frontier comparison')
print(f'  true frontier loss      : {true_frontier_loss:.6f}')
print(f'  full-window prediction  : {full_window_frontier:.6f}')
print(f'  small-window prediction : {small_window_frontier:.6f}')
print()
print('two-regime sweep frontier comparison')
print(f'  true frontier loss      : {true_regime_frontier:.6f}')
print(f'  full-range prediction   : {full_regime_frontier:.6f}')
print(f'  local-window prediction : {local_regime_frontier:.6f}')


In [ ]:

RUN_STRETCH_TINY_SWEEP = False

def run_tiny_model_sweep() -> list[tuple[int, int, float]]:
    x_train = torch.linspace(-math.pi, math.pi, 128, dtype=torch.float32).unsqueeze(1)
    y_train = torch.sin(x_train) + 0.1 * torch.cos(3.0 * x_train)
    x_valid = torch.linspace(-3.0, 3.0, 64, dtype=torch.float32).unsqueeze(1)
    y_valid = torch.sin(x_valid) + 0.1 * torch.cos(3.0 * x_valid)

    results: list[tuple[int, int, float]] = []
    for width in (8, 16, 32, 64):
        torch.manual_seed(width)
        model = nn.Sequential(nn.Linear(1, width), nn.Tanh(), nn.Linear(width, 1))
        optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
        for _ in range(80):
            prediction = model(x_train)
            loss = torch.mean((prediction - y_train) ** 2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        with torch.no_grad():
            valid_loss = torch.mean((model(x_valid) - y_valid) ** 2).item()
        parameter_count = sum(parameter.numel() for parameter in model.parameters())
        results.append((width, parameter_count, valid_loss))
    return results

if RUN_STRETCH_TINY_SWEEP:
    stretch_results = run_tiny_model_sweep()
    print('stretch tiny sweep results (width, parameters, validation_loss):')
    for row in stretch_results:
        print(row)
else:
    print('Stretch tiny model sweep disabled by default. Set RUN_STRETCH_TINY_SWEEP = True to run it.')



## Assumptions

- The synthetic observations are clean enough that a single power-law family is identifiable over the measured window.
- The `6 N D` compute proxy is only a dense-transformer accounting heuristic, not a hardware-accurate wall-clock model.
- The bootstrap treats the observed sweep as representative of nearby perturbations in the same regime.
- The optional stretch sweep is a toy regression problem, not a language-model benchmark.



## Risks

Small-scale fits do not justify frontier extrapolation. A neat exponent on six tiny runs can fail at larger scale because the data mix, optimization recipe, architecture, tokenizer, context length, or bottleneck regime changed. Even if none of those change, a narrow observed window may simply hide the irreducible floor or a multi-regime curve.

The practical risk is organizational, not just mathematical: a smooth fit can make an expensive training decision feel more certain than the evidence warrants.



## Failure criteria

Treat this notebook as failed if any of the following happen:

- the synthetic exponent cannot be recovered within tolerance;
- the bootstrap interval is so unstable that it no longer covers the known exponent on this deterministic sweep;
- the compute helpers fail the inverse-consistency check;
- the ablations do not visibly show why local fits can mislead frontier extrapolation.



## Exercises

Work the companion exercise set in `exercises/research/23_scaling_laws_lab.md` before reading the solutions.

Suggested prompts:

1. Re-derive the log-linear fit when `L_inf` is known.
2. Explain why bootstrap width matters more than the point estimate alone.
3. Use `C ~= 6 N D` to compare two same-compute training recipes.
4. State one concrete condition under which you would refuse to extrapolate a small scaling fit.



## References

- Kaplan et al., *Scaling Laws for Neural Language Models* [@scaling_laws_for_neural_language_models_2020]
- Hoffmann et al., *Training Compute-Optimal Large Language Models* [@training_compute_optimal_large_language_models_2022]
